<a href="https://colab.research.google.com/github/AndresMontesDeOca/NLP_1/blob/main/Desafios/Desafio_4._AndresMontesDeOca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">

# **Procesamiento de Lenguaje Natural**
## **Desafio 4, Traductor**

### **Consigna**

* Replicar el modelo traductor desarrollado en clase y extender su entrenamiento utilizando un conjunto de datos más amplio y secuencias de mayor longitud.
* Modificar valores de hiperparámetros y analizar su impacto en el desempeño del traductor.
* Analizar el impacto del número de neuronas en las capas recurrentes.
* Generar y presentar al menos cinco ejemplos de traducciones producidas por el modelo entrenado.
* Interpretar a detalle los resultados obtenidos, considerando métricas de evaluación, calidad de las traducciones y posibles limitaciones.

## 1. Configuración del Entorno

In [3]:
import os
os.environ["WANDB_API_KEY"] = "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"



## 2. Importación de Librerías y W&B Login

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import Input, LSTM, GRU, Dense, Embedding, Dropout, Bidirectional
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import wandb

try:
    from wandb.keras import WandbMetricsLogger as WandbLogger
except ImportError:
    try:
        from wandb.integration.keras import WandbMetricsLogger as WandbLogger
    except ImportError:
        from wandb.keras import WandbCallback as WandbLogger

# 1. Intentamos obtener la clave usando los Secrets nativos de Google Colab
wandb_key = None
try:
    from google.colab import userdata
    wandb_key = userdata.get('WANDB_API_KEY')
    print("Clave leída exitosamente desde Colab Secrets.")
except Exception:
    pass

# 2. Si no estamos en Colab o falló, hacemos un Fallback al archivo .env local
if not wandb_key:
    try:
        from dotenv import load_dotenv
        load_dotenv('.env')
        load_dotenv('Desafios/.env')
        load_dotenv('../Desafios/.env')
        wandb_key = os.environ.get("WANDB_API_KEY")
        if wandb_key:
            print("Clave leída exitosamente desde .env local.")
    except ImportError:
        pass

# 3. Hacemos el Login definitivo
if wandb_key:
    wandb.login(key=wandb_key, relogin=True)
else:
    print("ERROR FATAL: No se encontró la WANDB_API_KEY ni en Colab Secrets ni en el .env local.")


## 3. Descarga y Preprocesamiento del Dataset

In [6]:
if not os.path.exists('spa-eng'):
    os.system("curl -L -o spa-eng.zip http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip")
    os.system("unzip -q spa-eng.zip")

with open("./spa-eng/spa.txt", encoding="utf-8") as f:
    lines = f.read().split("\n")[:-1]

MAX_NUM_SENTENCES = 30000

np.random.seed(42)
np.random.shuffle(lines)

input_sentences, output_sentences = [], []
for i, line in enumerate(lines):
    if i >= MAX_NUM_SENTENCES:
        break
    if '\t' not in line:
        continue
    input_sentence, output = line.rstrip().split('\t')[:2]
    input_sentences.append(input_sentence)
    output_sentences.append(output)


output_sentences_inputs = ['<sos> ' + s for s in output_sentences]
output_sentences_targets = [s + ' <eos>' for s in output_sentences]

MAX_VOCAB_SIZE = 10000

input_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
input_tokenizer.fit_on_texts(input_sentences)
input_integer_seq = input_tokenizer.texts_to_sequences(input_sentences)
word2idx_inputs = input_tokenizer.word_index

output_tokenizer = Tokenizer(
    num_words=MAX_VOCAB_SIZE, 
    filters='!"#$%&()*+,-./:;=¿?@[\\]^_`{|}~\t\n'
)
output_tokenizer.fit_on_texts(output_sentences_inputs + output_sentences_targets)
output_input_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_inputs)
output_target_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_targets)
word2idx_outputs = output_tokenizer.word_index

max_input_len = 14
max_out_len = 16

encoder_input_sequences = pad_sequences(input_integer_seq, maxlen=max_input_len)
decoder_input_sequences = pad_sequences(output_input_integer_seq, maxlen=max_out_len, padding='post')
decoder_output_sequences = pad_sequences(output_target_integer_seq, maxlen=max_out_len, padding='post')
num_words_output = min(len(word2idx_outputs) + 1, MAX_VOCAB_SIZE)


def make_dataset(enc_seqs, dec_in_seqs, dec_out_seqs, batch_size, num_classes):
    n = len(enc_seqs)
    def generator():
        for i in range(n):
            yield (
                enc_seqs[i].astype(np.int32),
                dec_in_seqs[i].astype(np.int32),
                dec_out_seqs[i].astype(np.int32),
            )

    def encode_one_hot(enc, dec_in, dec_out):
        y = tf.one_hot(dec_out, depth=num_classes)
        return (enc, dec_in), y

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(max_input_len,), dtype=tf.int32),
            tf.TensorSpec(shape=(max_out_len,),   dtype=tf.int32),
            tf.TensorSpec(shape=(max_out_len,),   dtype=tf.int32),
        )
    )
    ds = ds.map(encode_one_hot, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

BATCH_SIZE = 64
val_split = 0.15
split_idx = int(len(encoder_input_sequences) * (1 - val_split))

train_ds = make_dataset(
    encoder_input_sequences[:split_idx],
    decoder_input_sequences[:split_idx],
    decoder_output_sequences[:split_idx],
    BATCH_SIZE, num_words_output
)
val_ds = make_dataset(
    encoder_input_sequences[split_idx:],
    decoder_input_sequences[split_idx:],
    decoder_output_sequences[split_idx:],
    BATCH_SIZE, num_words_output
)


_PKL_PATH = 'gloveembedding.pkl'
_FILE_ID = '1KY6avD5I1eI2dxQzMkR3WExwKwRq2g94'

def _is_valid_pickle(path):
    try:
        with open(path, 'rb') as f:
            head = f.read(20)
        return b'<html' not in head.lower() and b'<!doctype' not in head.lower()
    except Exception:
        return False

if os.path.exists(_PKL_PATH) and not _is_valid_pickle(_PKL_PATH):
    os.remove(_PKL_PATH)

if not os.path.exists(_PKL_PATH):
    try:
        import gdown
        gdown.download(id=_FILE_ID, output=_PKL_PATH, quiet=True)
    except ImportError:
        os.system(f"curl -L -o {_PKL_PATH} 'https://drive.google.com/u/0/uc?id={_FILE_ID}&export=download&confirm=t'")

max_bytes = 2**28 - 1
raw = bytearray()
sz = os.path.getsize(_PKL_PATH)
with open(_PKL_PATH, 'rb') as f:
    for _ in range(0, sz, max_bytes):
        raw += f.read(max_bytes)
glove_embeddings = pickle.loads(raw)

idx_array = np.arange(glove_embeddings.shape[0])
glove_word2idx = dict(zip(glove_embeddings['word'], idx_array))

EMBED_DIM = 50
nb_words = min(MAX_VOCAB_SIZE, len(word2idx_inputs))
embedding_matrix = np.zeros((nb_words, EMBED_DIM))

for word, i in word2idx_inputs.items():
    if i < nb_words:
        idx_glove = glove_word2idx.get(word, -1)
        if idx_glove != -1:
            embedding_matrix[i] = glove_embeddings[idx_glove]['embedding']


output_sentences_inputs = ['<sos> ' + s for s in output_sentences]
output_sentences_targets = [s + ' <eos>' for s in output_sentences]

MAX_VOCAB_SIZE = 10000

input_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
input_tokenizer.fit_on_texts(input_sentences)
input_integer_seq = input_tokenizer.texts_to_sequences(input_sentences)
word2idx_inputs = input_tokenizer.word_index

output_tokenizer = Tokenizer(
    num_words=MAX_VOCAB_SIZE, 
    filters='!"#$%&()*+,-./:;=¿?@[\\]^_`{|}~\t\n'
)
output_tokenizer.fit_on_texts(output_sentences_inputs + output_sentences_targets)
output_input_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_inputs)
output_target_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_targets)
word2idx_outputs = output_tokenizer.word_index

max_input_len = 14
max_out_len = 16

encoder_input_sequences = pad_sequences(input_integer_seq, maxlen=max_input_len)
decoder_input_sequences = pad_sequences(output_input_integer_seq, maxlen=max_out_len, padding='post')
decoder_output_sequences = pad_sequences(output_target_integer_seq, maxlen=max_out_len, padding='post')
num_words_output = min(len(word2idx_outputs) + 1, MAX_VOCAB_SIZE)


def make_dataset(enc_seqs, dec_in_seqs, dec_out_seqs, batch_size, num_classes):
    n = len(enc_seqs)
    def generator():
        for i in range(n):
            yield (
                enc_seqs[i].astype(np.int32),
                dec_in_seqs[i].astype(np.int32),
                dec_out_seqs[i].astype(np.int32),
            )

    def encode_one_hot(enc, dec_in, dec_out):
        y = tf.one_hot(dec_out, depth=num_classes)
        return (enc, dec_in), y

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(max_input_len,), dtype=tf.int32),
            tf.TensorSpec(shape=(max_out_len,),   dtype=tf.int32),
            tf.TensorSpec(shape=(max_out_len,),   dtype=tf.int32),
        )
    )
    ds = ds.map(encode_one_hot, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

BATCH_SIZE = 64
val_split = 0.15
split_idx = int(len(encoder_input_sequences) * (1 - val_split))

train_ds = make_dataset(
    encoder_input_sequences[:split_idx],
    decoder_input_sequences[:split_idx],
    decoder_output_sequences[:split_idx],
    BATCH_SIZE, num_words_output
)
val_ds = make_dataset(
    encoder_input_sequences[split_idx:],
    decoder_input_sequences[split_idx:],
    decoder_output_sequences[split_idx:],
    BATCH_SIZE, num_words_output
)


_PKL_PATH = 'gloveembedding.pkl'
_FILE_ID = '1KY6avD5I1eI2dxQzMkR3WExwKwRq2g94'

def _is_valid_pickle(path):
    try:
        with open(path, 'rb') as f:
            head = f.read(20)
        return b'<html' not in head.lower() and b'<!doctype' not in head.lower()
    except Exception:
        return False

if os.path.exists(_PKL_PATH) and not _is_valid_pickle(_PKL_PATH):
    os.remove(_PKL_PATH)

if not os.path.exists(_PKL_PATH):
    try:
        import gdown
        gdown.download(id=_FILE_ID, output=_PKL_PATH, quiet=True)
    except ImportError:
        os.system(f"curl -L -o {_PKL_PATH} 'https://drive.google.com/u/0/uc?id={_FILE_ID}&export=download&confirm=t'")

max_bytes = 2**28 - 1
raw = bytearray()
sz = os.path.getsize(_PKL_PATH)
with open(_PKL_PATH, 'rb') as f:
    for _ in range(0, sz, max_bytes):
        raw += f.read(max_bytes)
glove_embeddings = pickle.loads(raw)

idx_array = np.arange(glove_embeddings.shape[0])
glove_word2idx = dict(zip(glove_embeddings['word'], idx_array))

EMBED_DIM = 50
nb_words = min(MAX_VOCAB_SIZE, len(word2idx_inputs))
embedding_matrix = np.zeros((nb_words, EMBED_DIM))

for word, i in word2idx_inputs.items():
    if i < nb_words:
        idx_glove = glove_word2idx.get(word, -1)
        if idx_glove != -1:
            embedding_matrix[i] = glove_embeddings[idx_glove]['embedding']

## 4. Carga de Embeddings Pre-entrenados (GloVe)

In [7]:
output_sentences_inputs = ['<sos> ' + s for s in output_sentences]
output_sentences_targets = [s + ' <eos>' for s in output_sentences]

MAX_VOCAB_SIZE = 10000

input_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
input_tokenizer.fit_on_texts(input_sentences)
input_integer_seq = input_tokenizer.texts_to_sequences(input_sentences)
word2idx_inputs = input_tokenizer.word_index

output_tokenizer = Tokenizer(
    num_words=MAX_VOCAB_SIZE, 
    filters='!"#$%&()*+,-./:;=¿?@[\\]^_`{|}~\t\n'
)
output_tokenizer.fit_on_texts(output_sentences_inputs + output_sentences_targets)
output_input_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_inputs)
output_target_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_targets)
word2idx_outputs = output_tokenizer.word_index

max_input_len = 14
max_out_len = 16

encoder_input_sequences = pad_sequences(input_integer_seq, maxlen=max_input_len)
decoder_input_sequences = pad_sequences(output_input_integer_seq, maxlen=max_out_len, padding='post')
decoder_output_sequences = pad_sequences(output_target_integer_seq, maxlen=max_out_len, padding='post')
num_words_output = min(len(word2idx_outputs) + 1, MAX_VOCAB_SIZE)


def make_dataset(enc_seqs, dec_in_seqs, dec_out_seqs, batch_size, num_classes):
    n = len(enc_seqs)
    def generator():
        for i in range(n):
            yield (
                enc_seqs[i].astype(np.int32),
                dec_in_seqs[i].astype(np.int32),
                dec_out_seqs[i].astype(np.int32),
            )

    def encode_one_hot(enc, dec_in, dec_out):
        y = tf.one_hot(dec_out, depth=num_classes)
        return (enc, dec_in), y

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(max_input_len,), dtype=tf.int32),
            tf.TensorSpec(shape=(max_out_len,),   dtype=tf.int32),
            tf.TensorSpec(shape=(max_out_len,),   dtype=tf.int32),
        )
    )
    ds = ds.map(encode_one_hot, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

BATCH_SIZE = 64
val_split = 0.15
split_idx = int(len(encoder_input_sequences) * (1 - val_split))

train_ds = make_dataset(
    encoder_input_sequences[:split_idx],
    decoder_input_sequences[:split_idx],
    decoder_output_sequences[:split_idx],
    BATCH_SIZE, num_words_output
)
val_ds = make_dataset(
    encoder_input_sequences[split_idx:],
    decoder_input_sequences[split_idx:],
    decoder_output_sequences[split_idx:],
    BATCH_SIZE, num_words_output
)


_PKL_PATH = 'gloveembedding.pkl'
_FILE_ID = '1KY6avD5I1eI2dxQzMkR3WExwKwRq2g94'

def _is_valid_pickle(path):
    try:
        with open(path, 'rb') as f:
            head = f.read(20)
        return b'<html' not in head.lower() and b'<!doctype' not in head.lower()
    except Exception:
        return False

if os.path.exists(_PKL_PATH) and not _is_valid_pickle(_PKL_PATH):
    os.remove(_PKL_PATH)

if not os.path.exists(_PKL_PATH):
    try:
        import gdown
        gdown.download(id=_FILE_ID, output=_PKL_PATH, quiet=True)
    except ImportError:
        os.system(f"curl -L -o {_PKL_PATH} 'https://drive.google.com/u/0/uc?id={_FILE_ID}&export=download&confirm=t'")

max_bytes = 2**28 - 1
raw = bytearray()
sz = os.path.getsize(_PKL_PATH)
with open(_PKL_PATH, 'rb') as f:
    for _ in range(0, sz, max_bytes):
        raw += f.read(max_bytes)
glove_embeddings = pickle.loads(raw)

idx_array = np.arange(glove_embeddings.shape[0])
glove_word2idx = dict(zip(glove_embeddings['word'], idx_array))

EMBED_DIM = 50
nb_words = min(MAX_VOCAB_SIZE, len(word2idx_inputs))
embedding_matrix = np.zeros((nb_words, EMBED_DIM))

for word, i in word2idx_inputs.items():
    if i < nb_words:
        idx_glove = glove_word2idx.get(word, -1)
        if idx_glove != -1:
            embedding_matrix[i] = glove_embeddings[idx_glove]['embedding']

<>:13: SyntaxWarning: invalid escape sequence '\]'
<>:13: SyntaxWarning: invalid escape sequence '\]'
/tmp/ipykernel_3633/550555031.py:13: SyntaxWarning: invalid escape sequence '\]'
  filters='!"#$%&()*+,-./:;=¿?@[\]^_`{|}~\t\n'


## 5. Arquitectura del Modelo e Inferencia

In [10]:
def build_seq2seq_model(n_units=256, dropout_rate=0.3, rnn_type='lstm', bidirectional=False):
    enc_inputs = Input(shape=(max_input_len,), name='encoder_inputs')
    
    enc_emb_layer = Embedding(
        input_dim=nb_words,
        output_dim=EMBED_DIM,
        weights=[embedding_matrix],
        trainable=False,
        name='encoder_embedding'
    )
    enc_emb = Dropout(dropout_rate, name='encoder_dropout')(enc_emb_layer(enc_inputs))
    
    if rnn_type == 'gru':
        enc_rnn_layer = GRU(n_units, return_state=True, name='encoder_gru')
        if bidirectional:
            enc_rnn_layer = Bidirectional(enc_rnn_layer, name='encoder_bilstm')
            enc_out, forward_h, backward_h = enc_rnn_layer(enc_emb)
            state_h = tf.keras.layers.Concatenate()([forward_h, backward_h])
            enc_states = [state_h]
            dec_units = n_units * 2
        else:
            _, state_h = enc_rnn_layer(enc_emb)
            enc_states = [state_h]
            dec_units = n_units
    else:
        enc_rnn_layer = LSTM(n_units, return_state=True, name='encoder_lstm')
        if bidirectional:
            enc_rnn_layer = Bidirectional(enc_rnn_layer, name='encoder_bilstm')
            enc_out, forward_h, forward_c, backward_h, backward_c = enc_rnn_layer(enc_emb)
            state_h = tf.keras.layers.Concatenate()([forward_h, backward_h])
            state_c = tf.keras.layers.Concatenate()([forward_c, backward_c])
            enc_states = [state_h, state_c]
            dec_units = n_units * 2
        else:
            _, state_h, state_c = enc_rnn_layer(enc_emb)
            enc_states = [state_h, state_c]
            dec_units = n_units
            
    dec_inputs = Input(shape=(max_out_len,), name='decoder_inputs')
    
    dec_emb_layer = Embedding(
        input_dim=num_words_output,
        output_dim=dec_units, 
        name='decoder_embedding'
    )
    dec_emb = Dropout(dropout_rate, name='decoder_dropout')(dec_emb_layer(dec_inputs))
    
    if rnn_type == 'gru':
        dec_rnn_layer = GRU(dec_units, return_sequences=True, return_state=True, name='decoder_gru')
        dec_out, _ = dec_rnn_layer(dec_emb, initial_state=enc_states)
    else:
        dec_rnn_layer = LSTM(dec_units, return_sequences=True, return_state=True, name='decoder_lstm')
        dec_out, _, _ = dec_rnn_layer(dec_emb, initial_state=enc_states)
        
    dec_dense_layer = Dense(num_words_output, activation='softmax', name='decoder_dense')
    dec_outputs = dec_dense_layer(dec_out)
    
    model = Model([enc_inputs, dec_inputs], dec_outputs)
    
    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        metrics=['accuracy']
    )
    
    return model, enc_inputs, enc_states, enc_emb_layer, enc_rnn_layer, dec_inputs, dec_emb_layer, dec_rnn_layer, dec_dense_layer


# --- Funciones de Inferencia ---
def build_encoder_inference(enc_inputs, enc_emb_layer, enc_rnn_layer, rnn_type='lstm', bidirectional=False):
    enc_emb = enc_emb_layer(enc_inputs)
    if rnn_type == 'gru':
        if bidirectional:
            enc_out, forward_h, backward_h = enc_rnn_layer(enc_emb)
            state_h = tf.keras.layers.Concatenate()([forward_h, backward_h])
            return Model(enc_inputs, [state_h])
        else:
            _, state_h = enc_rnn_layer(enc_emb)
            return Model(enc_inputs, [state_h])
    else:
        if bidirectional:
            enc_out, forward_h, forward_c, backward_h, backward_c = enc_rnn_layer(enc_emb)
            state_h = tf.keras.layers.Concatenate()([forward_h, backward_h])
            state_c = tf.keras.layers.Concatenate()([forward_c, backward_c])
            return Model(enc_inputs, [state_h, state_c])
        else:
            _, state_h, state_c = enc_rnn_layer(enc_emb)
            return Model(enc_inputs, [state_h, state_c])

def build_decoder_inference(dec_emb_layer, dec_rnn_layer, dec_dense_layer, dec_units, rnn_type='lstm'):
    dec_input_single = Input(shape=(1,), name='dec_input_single')
    dec_emb_single = dec_emb_layer(dec_input_single)
    
    if rnn_type == 'gru':
        dec_state_h_in = Input(shape=(dec_units,), name='dec_state_h')
        dec_out, h_out = dec_rnn_layer(dec_emb_single, initial_state=[dec_state_h_in])
        dec_out = dec_dense_layer(dec_out)
        return Model([dec_input_single, dec_state_h_in], [dec_out, h_out])
    else:
        dec_state_h_in = Input(shape=(dec_units,), name='dec_state_h')
        dec_state_c_in = Input(shape=(dec_units,), name='dec_state_c')
        dec_out, h_out, c_out = dec_rnn_layer(dec_emb_single, initial_state=[dec_state_h_in, dec_state_c_in])
        dec_out = dec_dense_layer(dec_out)
        return Model([dec_input_single, dec_state_h_in, dec_state_c_in], [dec_out, h_out, c_out])

def translate_sentence(input_seq, encoder_model, decoder_model, rnn_type='lstm'):
    states = encoder_model.predict(input_seq, verbose=0)
    if rnn_type == 'gru':
        h = states
    else:
        h, c = states
        
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = word2idx_outputs['<sos>']
    eos = word2idx_outputs['<eos>']
    
    idx2word_target = {v: k for k, v in word2idx_outputs.items()}
    output_sentence = []
    
    for _ in range(max_out_len):
        if rnn_type == 'gru':
            output_tokens, h = decoder_model.predict([target_seq, h], verbose=0)
        else:
            output_tokens, h, c = decoder_model.predict([target_seq, h, c], verbose=0)
            
        idx = np.argmax(output_tokens[0, 0, :])
        if idx == eos:
            break
        if idx > 0:
            output_sentence.append(idx2word_target.get(idx, ''))
        target_seq[0, 0] = idx

    return ' '.join(output_sentence)

def translate(text, encoder_model, decoder_model, rnn_type='lstm'):
    seq = input_tokenizer.texts_to_sequences([text])
    seq = pad_sequences(seq, maxlen=max_input_len)
    return translate_sentence(seq, encoder_model, decoder_model, rnn_type)


def run_experiment(exp_name, n_units=256, rnn_type='lstm', bidirectional=False):
    os.environ["WANDB_SILENT"] = "true"
    
    wandb_key = os.environ.get("WANDB_API_KEY")
    if wandb_key:
        wandb.login(key=wandb_key, relogin=True)
    else:
        wandb.login(anonymous="allow")

    run = wandb.init(
        project="Desafio4_NLP_Traductor",
        name=exp_name,
        config={
            "learning_rate": 5e-4,
            "epochs": 30,
            "batch_size": BATCH_SIZE,
            "n_units": n_units,
            "rnn_type": rnn_type,
            "bidirectional": bidirectional,
            "dataset_size": MAX_NUM_SENTENCES
        },
        reinit=True
    )

    model, *components = build_seq2seq_model(n_units=n_units, rnn_type=rnn_type, bidirectional=bidirectional)
    
    lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5, verbose=1)
    early_stop = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1)
    
    try:
        wandb_logger = WandbLogger()
        callbacks = [lr_scheduler, early_stop, wandb_logger]
    except NameError:
        callbacks = [lr_scheduler, early_stop]

    hist = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=wandb.config.epochs,
        callbacks=callbacks,
        verbose=1
    )
    wandb.finish()
    
    # --- Evaluación Automática del Modelo Recién Entrenado ---
    print(f"\\n=============================================")
    print(f"  Resultados de Inferencia: {exp_name}")
    print(f"=============================================")
    
    enc_in, enc_states, enc_emb, enc_rnn, dec_in, dec_emb, dec_rnn, dec_dense = components
    dec_units = n_units * 2 if bidirectional else n_units
    
    encoder_model = build_encoder_inference(enc_in, enc_emb, enc_rnn, rnn_type=rnn_type, bidirectional=bidirectional)
    decoder_model = build_decoder_inference(dec_emb, dec_rnn, dec_dense, dec_units=dec_units, rnn_type=rnn_type)
    
    frases_prueba = [
        "I want to eat an apple.",
        "She is reading a very good book.",
        "What time does the train leave?",
        "We went to the beach yesterday.",
        "The weather is beautiful today, isn't it?"
    ]
    
    for s in frases_prueba:
        print(f"EN: {s}")
        print(f"ES: {translate(s, encoder_model, decoder_model, rnn_type=rnn_type)}\\n")
        
    return model, components

## 6. Experimentos de Entrenamiento

In [11]:
model_1, comp_1 = run_experiment("Exp1_LSTM_256", n_units=256, rnn_type='lstm', bidirectional=False)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 1/30
    397/Unknown 17s 32ms/step - accuracy: 0.5778 - loss: 4.0322

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


399/399 ━━━━━━━━━━━━━━━━━━━━ 19s 37ms/step - accuracy: 0.6161 - loss: 2.9614 - val_accuracy: 0.6479 - val_loss: 2.4208 - learning_rate: 5.0000e-04
Epoch 2/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 15s 36ms/step - accuracy: 0.6478 - loss: 2.4128 - val_accuracy: 0.6600 - val_loss: 2.2669 - learning_rate: 5.0000e-04
Epoch 3/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 15s 37ms/step - accuracy: 0.6624 - loss: 2.2497 - val_accuracy: 0.6747 - val_loss: 2.1279 - learning_rate: 5.0000e-04
Epoch 4/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 15s 37ms/step - accuracy: 0.6747 - loss: 2.1060 - val_accuracy: 0.6849 - val_loss: 2.0073 - learning_rate: 5.0000e-04
Epoch 5/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 15s 37ms/step - accuracy: 0.6861 - loss: 1.9690 - val_accuracy: 0.6955 - val_loss: 1.8956 - learning_rate: 5.0000e-04
Epoch 6/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 15s 37ms/step - accuracy: 0.6965 - loss: 1.8503 - val_accuracy: 0.7035 - val_loss: 1.8100 - learning_rate: 5.0000e-04
Epoch 7/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 15s 37ms/step - accurac

epoch/accuracy,▁▂▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▆▅▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
epoch/val_accuracy,▁▂▃▃▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇█████████
epoch/val_loss,█▇▆▅▅▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,0.83138
epoch/epoch,29
epoch/learning_rate,0.0005
epoch/loss,0.73874
epoch/val_accuracy,0.76529


### Experimento 2: LSTM (512 unidades)

In [12]:
model_2, comp_2 = run_experiment("Exp2_LSTM_512", n_units=512, rnn_type='lstm', bidirectional=False)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Epoch 1/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 21s 53ms/step - accuracy: 0.7666 - loss: 1.1046 - val_accuracy: 0.7529 - val_loss: 1.3653 - learning_rate: 5.0000e-04
Epoch 11/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 21s 53ms/step - accuracy: 0.7777 - loss: 1.0218 - val_accuracy: 0.7568 - val_loss: 1.3385 - learning_rate: 5.0000e-04
Epoch 12/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 21s 53ms/step - accuracy: 0.7897 - loss: 0.9483 - val_accuracy: 0.7600 - val_loss: 1.3179 - learning_rate: 5.0000e-04
Epoch 13/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 21s 53ms/step - accuracy: 0.8021 - loss: 0.8795 - val_accuracy: 0.7616 - val_loss: 1.3034 - learning_rate: 5.0000e-04
Epoch 14/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 21s 53ms/step - accuracy: 0.8146 - loss: 0.8193 - val_accuracy: 0.7641 - val_loss: 1.2925 - learning_rate: 5.0000e-04
Epoch 15/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 21s 53ms/step - accuracy: 0.8260 - loss: 0.7634 - val_accuracy: 0.7668 - val_loss: 1.2846 - learning_rate: 5.0000e-04
Epoch 16/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 21s 53

epoch/accuracy,▁▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇█████
epoch/epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
epoch/learning_rate,███████████████████▄▄▄▂▂▁
epoch/loss,█▇▆▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁
epoch/val_accuracy,▁▂▃▄▅▅▆▆▆▇▇▇▇▇▇██████████
epoch/val_loss,█▆▅▄▄▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,0.89751
epoch/epoch,24
epoch/learning_rate,6e-05
epoch/loss,0.44151
epoch/val_accuracy,0.77786


### Experimento 3: GRU (256 unidades)

In [13]:
model_3, comp_3 = run_experiment("Exp3_GRU_256", n_units=256, rnn_type='gru', bidirectional=False)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Epoch 1/30
    399/Unknown 15s 33ms/step - accuracy: 0.5844 - loss: 4.0527

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


399/399 ━━━━━━━━━━━━━━━━━━━━ 17s 38ms/step - accuracy: 0.6206 - loss: 2.9600 - val_accuracy: 0.6484 - val_loss: 2.3739 - learning_rate: 5.0000e-04
Epoch 2/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 15s 38ms/step - accuracy: 0.6527 - loss: 2.3249 - val_accuracy: 0.6670 - val_loss: 2.1550 - learning_rate: 5.0000e-04
Epoch 3/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 15s 38ms/step - accuracy: 0.6727 - loss: 2.0997 - val_accuracy: 0.6846 - val_loss: 1.9783 - learning_rate: 5.0000e-04
Epoch 4/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 15s 37ms/step - accuracy: 0.6877 - loss: 1.9211 - val_accuracy: 0.6982 - val_loss: 1.8497 - learning_rate: 5.0000e-04
Epoch 5/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 15s 37ms/step - accuracy: 0.7001 - loss: 1.7774 - val_accuracy: 0.7076 - val_loss: 1.7482 - learning_rate: 5.0000e-04
Epoch 6/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 15s 37ms/step - accuracy: 0.7097 - loss: 1.6555 - val_accuracy: 0.7154 - val_loss: 1.6676 - learning_rate: 5.0000e-04
Epoch 7/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 15s 37ms/step - accurac

epoch/accuracy,▁▂▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇█████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
epoch/learning_rate,█████████████████████████▁▁▁▁▁
epoch/loss,█▆▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▂▃▄▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇██████████
epoch/val_loss,█▇▅▅▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,0.85915
epoch/epoch,29
epoch/learning_rate,0.00025
epoch/loss,0.61444
epoch/val_accuracy,0.77026


### Experimento 4: Bidirectional LSTM (256 unidades)

In [14]:
model_4, comp_4 = run_experiment("Exp4_BiLSTM_256", n_units=256, rnn_type='lstm', bidirectional=True)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Epoch 1/30
    398/Unknown 22s 50ms/step - accuracy: 0.5985 - loss: 3.4917

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


399/399 ━━━━━━━━━━━━━━━━━━━━ 25s 56ms/step - accuracy: 0.6334 - loss: 2.7207 - val_accuracy: 0.6609 - val_loss: 2.2892 - learning_rate: 5.0000e-04
Epoch 2/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 22s 55ms/step - accuracy: 0.6649 - loss: 2.2298 - val_accuracy: 0.6809 - val_loss: 2.0586 - learning_rate: 5.0000e-04
Epoch 3/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 22s 55ms/step - accuracy: 0.6826 - loss: 2.0051 - val_accuracy: 0.6945 - val_loss: 1.8918 - learning_rate: 5.0000e-04
Epoch 4/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 22s 55ms/step - accuracy: 0.6966 - loss: 1.8265 - val_accuracy: 0.7057 - val_loss: 1.7654 - learning_rate: 5.0000e-04
Epoch 5/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 22s 55ms/step - accuracy: 0.7085 - loss: 1.6762 - val_accuracy: 0.7165 - val_loss: 1.6671 - learning_rate: 5.0000e-04
Epoch 6/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 22s 55ms/step - accuracy: 0.7193 - loss: 1.5477 - val_accuracy: 0.7241 - val_loss: 1.5910 - learning_rate: 5.0000e-04
Epoch 7/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 22s 55ms/step - accurac

epoch/accuracy,▁▂▂▃▃▃▄▄▄▅▅▅▆▆▆▆▇▇▇▇███
epoch/epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇██
epoch/learning_rate,████████████████████▃▃▁
epoch/loss,█▆▆▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
epoch/val_accuracy,▁▂▃▄▅▅▅▆▆▆▇▇▇▇▇▇███████
epoch/val_loss,█▆▅▄▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,0.87409
epoch/epoch,22
epoch/learning_rate,0.00013
epoch/loss,0.54283
epoch/val_accuracy,0.77089


### **Análisis de Resultados y Conclusiones**

#### 1. Impacto del número de neuronas y tipos de RNN
* **LSTM 256 vs 512:** 
* **LSTM vs GRU:** 
* **Bidirectional LSTM:** 

#### 2. Calidad de las Traducciones
* **Aciertos:** 
* **Limitaciones:** 

#### 3. Conclusión sobre el Escalamiento de Datos
* 